## CPSC 4970 Module 3 Homework - Jonathan Braun

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from math import sqrt

In [2]:
train_url = "https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3train.csv"
test_url = "https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3test.csv"

training_data = pd.read_csv(train_url)
testing_data = pd.read_csv(test_url)

display(training_data.head())
display(testing_data.head())

,A,B,C,D,E,F,G,H,I,J,K,L,M,N
0,0.003555,92.159562,6.658918,0.0,1.994330,35.960534,290.003375,20.973961,7.523349,2410.680571,33.715837,1965.274105,34.370406,220.134692
1,0.015362,0.000000,20.380324,0.0,1.738552,35.118264,350.939667,25.471824,15.046698,1970.894250,39.224961,1965.274105,63.081429,198.121223
2,0.015351,0.000000,20.380324,0.0,1.738552,39.296796,271.766966,25.471824,15.046698,1970.894250,39.224961,1945.121257,27.813803,318.278075
3,0.018208,0.000000,6.284173,0.0,1.697776,38.274040,203.714027,31.087615,22.570047,1808.010428,41.208245,1954.034064,20.290963,306.354113
4,0.038841,0.000000,6.284173,0.0,1.697776,39.088964,241.076425,31.087615,22.570047,1808.010428,41.208245,1965.274105,36.785997,332.036493


,A,B,C,D,E,F,G,H,I,J,K,L,M
0,0.042205,168.959198,6.284173,0.0,1.749673,40.582078,319.804335,15.893032,52.663442,1808.010428,40.54715,1965.274105,44.653922
1,0.027743,168.959198,6.284173,0.0,1.749673,37.459117,312.687688,16.321229,52.663442,1808.010428,40.54715,1965.274105,51.969711
2,0.277305,0.000000,28.538219,0.0,2.016572,36.288691,366.952123,17.012498,30.093396,2475.834100,40.54715,1965.274105,31.333664
3,0.196540,0.000000,28.538219,0.0,2.016572,32.662556,341.154277,15.909954,30.093396,2475.834100,40.54715,1962.006076,68.809830
4,1.482478,0.000000,28.538219,0.0,2.016572,27.198743,168.130791,12.919755,30.093396,2475.834100,40.54715,1735.274150,87.237337


In [3]:
training_data.info()

testing_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       306 non-null    float64
 1   B       306 non-null    float64
 2   C       306 non-null    float64
 3   D       306 non-null    float64
 4   E       306 non-null    float64
 5   F       306 non-null    float64
 6   G       306 non-null    float64
 7   H       306 non-null    float64
 8   I       306 non-null    float64
 9   J       306 non-null    float64
 10  K       306 non-null    float64
 11  L       306 non-null    float64
 12  M       306 non-null    float64
 13  N       306 non-null    float64
dtypes: float64(14)
memory usage: 33.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       100 non-null    float64
 1   B       100 non-null    float64
 2   C       100 non-null    fl

In [4]:
training_data = training_data.dropna()

X = training_data.iloc[:, :-1]
y = training_data.iloc[:, -1]

display(X.head())
display(y.head())

,A,B,C,D,E,F,G,H,I,J,K,L,M
0,0.003555,92.159562,6.658918,0.0,1.994330,35.960534,290.003375,20.973961,7.523349,2410.680571,33.715837,1965.274105,34.370406
1,0.015362,0.000000,20.380324,0.0,1.738552,35.118264,350.939667,25.471824,15.046698,1970.894250,39.224961,1965.274105,63.081429
2,0.015351,0.000000,20.380324,0.0,1.738552,39.296796,271.766966,25.471824,15.046698,1970.894250,39.224961,1945.121257,27.813803
3,0.018208,0.000000,6.284173,0.0,1.697776,38.274040,203.714027,31.087615,22.570047,1808.010428,41.208245,1954.034064,20.290963
4,0.038841,0.000000,6.284173,0.0,1.697776,39.088964,241.076425,31.087615,22.570047,1808.010428,41.208245,1965.274105,36.785997


0    220.134692
1    198.121223
2    318.278075
3    306.354113
4    332.036493
Name: N, dtype: float64

The training data is labeled, with the final column used as the target variable. Rows with missing values are removed using `dropna()`. The testing data is not used during training because it does not include the target column.

## Cross-Validated Model

In [5]:
pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scale", StandardScaler()),
    ("regr", TransformedTargetRegressor(transformer=StandardScaler()))
])

param_grid = [
    {
        "poly__degree": [1, 2, 3, 4, 5, 6],
        "regr__regressor": [LinearRegression()]
    },
    {
        "poly__degree": [1, 2, 3, 4, 5, 6],
        "regr__regressor": [Ridge()],
        "regr__regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        "poly__degree": [1, 2, 3, 4, 5, 6],
        "regr__regressor": [Lasso(max_iter=100000)],
        "regr__regressor__alpha": [0.001, 0.01, 0.1, 1, 10]
    }
]

model = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

model.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...rdScaler()))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'poly__degree': [1, 2, ...], 'regr__regressor': [LinearRegression()]}, {'poly__degree': [1, 2, ...], 'regr__regressor': [Ridge()], 'regr__regressor__alpha': [0.001, 0.01, ...]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cro